# Tutorial 1 — Build one LC resonator from primitives

## Prerequisites

Python and the idea that a capacitor and inductor connected in parallel
form an LC resonator. No SCNSim composite, optimizer, harmonic-balance,
or team façade knowledge is assumed.

## Objects introduced here

`units`, `CircuitPlan`, primitive capacitor and inductor components,
`ElectricNodeRef`, the canonical ground, a logical Port,
`CircuitDiagramSpec`, `CircuitRun`, `NetworkViewRef`,
`ReductionPipeline`, `DirectSolveSpec`, `SParameterTrace`,
`DiagonalRootSpec`, typed Results, and `ReportSpec`.

## What the reader will build

A one-port reflective resonator: a 50-ohm measurement boundary connects
through a coupling capacitor to a parallel capacitor/inductor whose
returns share the Plan ground.

## What inspection or Result is produced

An authoring diagram, a Direct S11 response Result, one diagonal-root
Result, and a Report that names both exact inputs. The current V1
scaffold deliberately stops at construction, before it can produce
numerical evidence.

## Start with units

Physical inputs carry units. `u` is the one SCNSim unit registry, so
`6.0 * u.fF` is a capacitance rather than an unqualified number.

In [ ]:
from scnsim import units as u

## Add only primitive parts

We need one coupling capacitor and the two branches of the resonator. A
`CircuitPlan` owns their topology; the built-in Library only creates
immutable parts for that Plan. `plan.add()` registers each created part
with the one Plan that will own its pins and parameter identity.

In [ ]:
from scnsim import CircuitPlan, library as sc

plan = CircuitPlan(id="simple_resonator")
coupling_cap = plan.add(
    sc.capacitor(id="coupling_cap", capacitance=6.0 * u.fF)
)
resonator_cap = plan.add(
    sc.capacitor(id="resonator_cap", capacitance=110.0 * u.fF)
)
resonator_inductor = plan.add(
    sc.inductor(id="resonator_inductor", inductance=5.8 * u.nH)
)

## State every electrical connection

Parts alone do not say which terminals are equipotential. `net()`
receives a complete group of pins and returns the node handle. The Plan
already owns its one canonical ground, so ordinary primitive returns
attach with `ground()`; we never declare another ground component or a
node called `"ground"`.

In [ ]:
signal_boundary = plan.net(coupling_cap.pin("terminal_1"))
resonator_node = plan.net(
    coupling_cap.pin("terminal_2"),
    resonator_cap.pin("terminal_1"),
    resonator_inductor.pin("terminal_1"),
    id="resonator_node",
)
plan.ground(
    resonator_cap.pin("terminal_2"),
    resonator_inductor.pin("terminal_2"),
)

## Attach the measurement boundary as a logical Port

The electrical node is already complete. A logical Port adds the
external 50-ohm wave/load boundary to that node; it does not make
another wire or another physical degree of freedom.

In [ ]:
signal_port = plan.add_port(
    id="signal_in",
    at=signal_boundary,
    role="terminated",
    reference_impedance=50.0 * u.ohm,
)

## Inspect the authored circuit before asking a physics question

`CircuitDiagramSpec` chooses the authoring view. Rendering validates and
materializes the diagram; `show()` only presents that materialized
inspection.

In [ ]:
from scnsim import CircuitDiagramSpec

diagram = plan.render_schematic(
    CircuitDiagramSpec(show_parameter_values=True)
)
diagram.show()

## Choose views from one sealed Run

A `CircuitRun` seals this one Plan to an evidence workspace. Its
`original` property is the unreduced `NetworkViewRef`; `retain()`
derives immutable views without solving. The response view retains the
port-promoted `signal_in` coordinate, while the quantity view retains
the named resonator coordinate.

In [ ]:
from scnsim import CircuitRun, ReductionPipeline

run = CircuitRun(plan=plan, workspace="results/simple_resonator")
original = run.original
response_view = original.reduce(
    ReductionPipeline().retain("signal_in")
)
quantity_view = original.reduce(
    ReductionPipeline().retain("resonator_node")
)

## Solve the reflection response

`DirectSolveSpec` requests the complete selected-view response over an
exact frequency grid. The named S11 trace is a projection of that one
solve, not a second analysis. `solve()` returns a typed Direct Result.

In [ ]:
from scnsim import DirectSolveSpec, SParameterTrace

frequency_grid = tuple(
    frequency * u.GHz
    for frequency in (5.5, 5.6, 5.7, 5.8, 5.9, 6.0, 6.1, 6.2, 6.3)
)
direct_spec = DirectSolveSpec(
    frequencies=frequency_grid,
    traces=(
        SParameterTrace(
            id="reflection",
            input_port="signal_in",
            input_mode=(),
            output_port="signal_in",
            output_mode=(),
        ),
    ),
)
direct = run.solve(response_view, direct_spec)
direct.s.show(magnitude="db")
direct.traces["reflection"].show(magnitude="db")

## Evaluate the resonator quantity directly

A plotted reflection minimum is not the same thing as a physical root.
`DiagonalRootSpec` identifies the resonance of the selected coordinate;
`root_hint` chooses the baseline branch rather than supplying a target.
`evaluate()` returns its own typed quantity Result.

In [ ]:
from scnsim import DiagonalRootSpec

root_spec = DiagonalRootSpec(
    coordinate="resonator_node",
    root_hint=6.0 * u.GHz,
)
root = run.evaluate(quantity_view, root_spec)
root.show()
root.frequency.to(u.GHz)

## Report the exact Results already produced

`ReportSpec` accepts explicit typed Results. `build_report()` assembles
them; it does not rerun the sweep or the root evaluation.

In [ ]:
from scnsim import ReportSpec

report = run.build_report(ReportSpec(inputs=(direct, root)))
report.show()

## Learned objects

You built a complete primitive `CircuitPlan`, grounded it explicitly,
attached a logical Port, derived `NetworkViewRef` values from one
`CircuitRun`, and requested typed Direct and quantity Results.

Next: [package the parallel L/C as one reusable
composite](../reusable_composite/01_composite_plan.qmd).